## Install Faker

Digunakan untuk menginstal library Faker pada lingkungan Colab. Library Faker digunakan untuk menghasilkan data tiruan seperti nama pelanggan, kota, dan tanggal transaksi.

In [1]:
!pip install faker -q

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_bersih.csv', 'transaksi_mentah_latihan.csv', 'transaksi_mentah.csv']


## Import Library dan Inisialisasi

Library yang digunakan terdiri dari NumPy untuk operasi numerik, Pandas untuk pengolahan data, Faker untuk menghasilkan data tiruan, dan Random untuk menentukan nilai secara acak.

In [3]:
import numpy as np
import pandas as pd
from faker import Faker
import random

## Membuat Dataset Sintetis

Membuat dataset transaksi mentah secara sintetis menggunakan library yang sudah diimpor. SEED = 42 digunakan agar proses pengacakan menghasilkan data yang konsisten ketika kode dijalankan kembali. Sebanyak 500 transaksi dibuat dengan atribut seperti ID transaksi, nama pelanggan, produk, kategori, harga, jumlah barang, metode pembayaran, tanggal transaksi, kota pengiriman, dan rating.
Data dibuat dengan variasi format pada beberapa kolom. Selain itu beberapa nilai kosong dan baris duplikat juga ditambahkan untuk mensimulasikan permasalahan umum yang ditemukan pada data transaksi nyata. Setelah data dibentuk menjadi DataFrame, dataset mentah disimpan dalam format CSV dengan nama transaksi_mentah.csv.


In [4]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "lite", "Max", "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f"{harga_dasar} ",
  ]
  harga = random.choice(harga_variants)

  # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

  rows.append({
      "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
      "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
      "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
  })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercata dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv(os.path.join(DIR_SIMPAN, "transaksi_mentah.csv"), index=False)
print("Dataset mentah tersimpan:", len(df), "baris")

Dataset mentah tersimpan: 515 baris


## Deteksi dan Penanganan Missing Value

Dilakukan perhitungan jumlah missing value pada setiap kolom dataFrame. Terdapat beberapa kolom yang memiliki missing value. Missing value ini perlu ditangani pada tahap pembersihan data agar dataset dapat digunakan untuk proses analisis selanjutnya.

In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


Dilakukan pembersihan data dengan menghapus data yang memiliki nilai kosong pada kolom customer_name dan payment_method. Sementara itu, nilai kosong pada kolom shipping_city diganti menjadi “Tidak Diketahui”.

In [6]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

## Deteksi dan Penanganan Duplicate

Dilakukan pengecekan data duplikat dan ditemukan 5 baris duplikat berdasarkan seluruh kolom maupun transaction_id. Data duplikat kemudian dihapus menggunakan drop_duplicates(), sehingga jumlah data bersih menjadi 490 baris.

In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## Koreksi Tipe Data dan Standardisasi Format

Dilakukan standardisasi pada data kategorikal dengan menghilangkan spasi yang tidak diperlukan dan menyamakan format penulisan menjadi Title Case. Untuk metode pembayaran COD, penulisannya dijadikan huruf kapital penuh.

In [8]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan: kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

Dilakukan pembersihan pada kolom price dengan menghapus simbol dan format yang tidak diperlukan, kemudian mengubah nilainya menjadi tipe numerik agar dapat digunakan dalam proses analisis.

In [9]:
# koreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Dilakukan standardisasi format tanggal pada kolom transaction_date menjadi YYYY-MM-DD agar penulisan tanggal konsisten dan lebih mudah digunakan dalam proses analisis.

In [10]:
# standardisasi format tanggal ke yyyy-mm-dd
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Dilakukan finalisasi tipe data dengan mengubah kolom quantity menjadi tipe integer dan price menjadi tipe float agar sesuai dengan jenis data yang digunakan dalam analisis.

In [11]:
# finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

## Ekspor Dataset Bersih

Dataset yang telah melalui proses pembersihan dan pra-pemrosesan disimpan dalam format CSV ke Drive sebagai transaksi_bersih.csv.

In [12]:
df.to_csv(os.path.join(DIR_SIMPAN, "transaksi_bersih.csv"), index=False)
print("File berhasil disimpan ke Drive.")

File berhasil disimpan ke Drive.


## Latihan

Latihan 1 -> ubah SEED menjadi 7

In [13]:
# Membuat dataset sintetis
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f"{harga_dasar} ",
  ]
  harga = random.choice(harga_variants)

  # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

  rows.append({
      "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
      "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
      "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
  })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercata dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv(os.path.join(DIR_SIMPAN, "transaksi_mentah_latihan.csv"), index=False)
print("Dataset mentah tersimpan:", len(df), "baris")

Dataset mentah tersimpan: 515 baris


In [14]:
# Deteksi dan Penanganan Missing Value
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [15]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [16]:
# Deteksi dan penanganan duplicate
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [17]:
# koreksi tipe data dan standardisasi format

# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan: kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [18]:
# koreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [19]:
# standardisasi format tanggal ke yyyy-mm-dd
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [20]:
# finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [23]:
# Ekspor data bersih
df.to_csv(os.path.join(DIR_SIMPAN, "transaksi_bersih_latihan.csv"), index=False)
print("File berhasil disimpan ke Drive.")

File berhasil disimpan ke Drive.


Jumlah baris pada dataset dengan SEED = 42 dan SEED = 7 berjumlah sama. Hal ini terjadi karena perubahan SEED hanya memengaruhi nilai acak yang dihasilkan, sedangkan jumlah data awal, jumlah missing value yang dibuat, dan jumlah duplikat tetap ditentukan oleh kode. Meskipun isi datanya berbeda, jumlah baris yang dihapus pada proses pembersihan menghasilkan jumlah akhir yang sama.

Latihan 2 -> tambahkan kolom `is_valid_price`

In [24]:
# menambahkan kolom validasi harga

df["is_valid_price"] = df["price"] > 0

# cek apakah ada harga yang tidak valid
print("Harga tidak valid:", (-df["is_valid_price"]).sum())

Harga tidak valid: 0


Hasil pengecekan menunjukkan tidak ada data dengan harga yang tidak valid.

Latihan 3 -> hitung jumlah transaksi per category dengan `value_counts`

In [25]:
jumlah_transaksi = df["category"].value_counts()

print("Jumlah transaksi per kategori:")
print(jumlah_transaksi)

Jumlah transaksi per kategori:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


Berdasarkan hasil perhitungan, kategori Rumah Tangga memiliki jumlah transaksi terbanyak yaitu 91 transaksi, sedangkan kategori Olahraga memiliki jumlah transaksi paling sedikit yaitu 74 transaksi.

## Studi Kasus

Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan bulanan: total transaksi yang dilaporkan tim IT (515) tidak sama dengan total yang dipakai tim Finance (490).

**1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda lakukan.**
Perbedaan jumlah transaksi terjadi karena kedua tim menggunakan data pada tahap yang berbeda. Tim IT menggunakan data mentah sebanyak 515 baris, yang terdiri dari 500 transaksi awal dan 15 baris duplikat. Setelah dilakukan proses preprocessing, seperti penghapusan data kosong pada kolom tertentu dan penghapusan duplikat, jumlah data menjadi 490 baris yang digunakan oleh Finance.

**2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity.**
490 baris tidak otomatis lebih benar dari 515 baris. Angka 490 merupakan data yang sudah melalui proses pembersihan, sedangkan 515 merupakan data mentah yang masih mengandung duplikat dan missing value. Dalam konsep Veracity, data yang telah dibersihkan cenderung lebih dapat dipercaya untuk analisis karena masalah kualitas datanya telah ditangani. Namun, proses pembersihan tetap harus didokumentasikan agar perbedaan jumlah data dapat dijelaskan.

**3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missingvalue kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?**
Kolom rating tetap memiliki missing value karena nilai yang kosong tidak diganti secara sembarangan agar tidak menciptakan rating yang sebenarnya tidak diberikan pelanggan. Untuk menghitung rata-rata rating, rata-rata sebaiknya dihitung dari transaksi yang memiliki rating lalu dilaporkan bersama jumlah data yang memiliki rating. Dengan demikian, hasil rata-rata tidak dianggap mewakili transaksi yang tidak memberikan rating.